In [1]:
from pathlib import Path

import duckdb
import pandas as pd

In [2]:
STATCAN_DIR = Path("data/external/statscan")
PROFILE_DIR = STATCAN_DIR / "profile"

TRACT_MAP_PATH = (
    STATCAN_DIR /
    "gta_store_census_tract_mapping.parquet"
)

OUTPUT_DIR = STATCAN_DIR / "filtered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find your StatsCan CSV

In [3]:
profile_files = (
    list(PROFILE_DIR.rglob("*.csv"))
    + list(PROFILE_DIR.rglob("*.CSV"))
)

for i, file in enumerate(profile_files):
    print(
        i,
        file.name,
        round(file.stat().st_size / 1024**2, 2),
        "MB"
    )

0 98-401-X2021007_English_CSV_data.csv 2512.42 MB
1 98-401-X2021007_Geo_starting_row.CSV 0.27 MB


In [4]:
PROFILE_PATH = profile_files[0]

print("Using:", PROFILE_PATH)

Using: data/external/statscan/profile/98-401-X2021007_eng_CSV/98-401-X2021007_English_CSV_data.csv


# Check whether this is CI file

In [6]:
if "CI_" in PROFILE_PATH.name or "2021007CI" in PROFILE_PATH.name:
    print("Confidence Interval Census Profile detected.")
    print(
        "We can still filter it, but basic variables such as "
        "Population 2021 may not be available."
    )
else:
    print("Standard Census Profile file detected.")

Standard Census Profile file detected.


# Walmart catchment tracts
# Cell 5 — Load only your relevant tract IDs

In [7]:
tract_map = pd.read_parquet(
    TRACT_MAP_PATH
)

print(tract_map.shape)
display(tract_map.head())

(1103, 6)


,store_id,store_name,DGUID,intersection_area_m2,tract_area_m2,area_weight
0,3106,Toronto Dufferin Mall,2021S05075350010.01,247461.063726,2.474611e+05,1.000000
1,3106,Toronto Dufferin Mall,2021S05075350010.02,537964.815388,5.379648e+05,1.000000
2,3106,Toronto Dufferin Mall,2021S05075350002.00,992806.909061,3.395314e+06,0.292405
3,3106,Toronto Dufferin Mall,2021S05075350003.00,958512.541558,9.585125e+05,1.000000
4,3106,Toronto Dufferin Mall,2021S05075350004.00,340251.728336,3.402517e+05,1.000000


# Get unique DGUIDs 

In [8]:
relevant_tracts = (
    tract_map["DGUID"]
    .astype(str)
    .dropna()
    .unique()
)

print(
    "Census tracts needed:",
    len(relevant_tracts)
)

print(relevant_tracts[:10])

Census tracts needed: 605
<ArrowStringArray>
['2021S05075350010.01', '2021S05075350010.02', '2021S05075350002.00',
 '2021S05075350003.00', '2021S05075350004.00', '2021S05075350005.00',
 '2021S05075350006.00', '2021S05075350009.00', '2021S05075350014.00',
 '2021S05075350015.00']
Length: 10, dtype: str


# DuckDB

In [9]:
con = duckdb.connect()

In [10]:
tract_df = pd.DataFrame({
    "DGUID": relevant_tracts
})

con.register(
    "relevant_tracts",
    tract_df
)

# Check CSV schema without pandas

In [17]:
schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_csv_auto(
    '{PROFILE_PATH}',
    encoding='latin-1',
    header=true,
    all_varchar=true,
    sample_size=100000
)
""").fetchdf()

display(schema)

,column_name,column_type,null,key,default,extra
0,CENSUS_YEAR,VARCHAR,YES,None,None,None
1,DGUID,VARCHAR,YES,None,None,None
2,ALT_GEO_CODE,VARCHAR,YES,None,None,None
3,GEO_LEVEL,VARCHAR,YES,None,None,None
4,GEO_NAME,VARCHAR,YES,None,None,None
5,TNR_SF,VARCHAR,YES,None,None,None
6,TNR_LF,VARCHAR,YES,None,None,None
7,DATA_QUALITY_FLAG,VARCHAR,YES,None,None,None
8,CHARACTERISTIC_ID,VARCHAR,YES,None,None,None
9,CHARACTERISTIC_NAME,VARCHAR,YES,None,None,None


# Extract only GTA Walmart catchments
# Cell 9 — Test tract matching

In [19]:
match_test = con.execute(f"""
SELECT
    COUNT(DISTINCT p.DGUID) AS profile_tracts_found
FROM read_csv_auto(
    '{PROFILE_PATH}',
    encoding='latin-1',
    header=true,
    all_varchar=true,
    sample_size=100000
) p

INNER JOIN relevant_tracts r
    ON p.DGUID = r.DGUID

WHERE p.GEO_LEVEL = 'Census tract'
""").fetchdf()

display(match_test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,profile_tracts_found
0,605


In [20]:
print(
    "Tracts we requested:",
    len(relevant_tracts)
)

Tracts we requested: 605


# Cell 10 — Filter relevant characteristics

In [21]:
candidate_path = (
    OUTPUT_DIR /
    "gta_census_demographic_candidates.parquet"
)

In [23]:
candidate_path = (
    OUTPUT_DIR /
    "gta_census_demographic_candidates.parquet"
)

con.execute(f"""
COPY (
    SELECT
        p.DGUID,
        p.GEO_NAME,
        p.CHARACTERISTIC_ID,
        p.CHARACTERISTIC_NAME,
        p.C1_COUNT_TOTAL

    FROM read_csv_auto(
        '{PROFILE_PATH}',
        encoding='latin-1',
        header=true,
        all_varchar=true,
        sample_size=100000
    ) p

    INNER JOIN relevant_tracts r
        ON p.DGUID = r.DGUID

    WHERE
        p.GEO_LEVEL = 'Census tract'

        AND (
            LOWER(p.CHARACTERISTIC_NAME) LIKE '%population%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%density%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%household%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%income%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%employment rate%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%renter%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%postsecondary%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%15 to 24%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%25 to 44%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%45 to 64%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%transportation and warehousing%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%manufacturing%'
            OR LOWER(p.CHARACTERISTIC_NAME) LIKE '%professional%'
        )
)
TO '{candidate_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

print("Saved:", candidate_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: data/external/statscan/filtered/gta_census_demographic_candidates.parquet


# Work with small Parquet now
# Cell 11 — Inspect filtered size

In [24]:
candidates = pd.read_parquet(
    candidate_path
)

print("Rows:", len(candidates))

print(
    "Tracts:",
    candidates["DGUID"].nunique()
)

print(
    "Characteristics:",
    candidates[
        "CHARACTERISTIC_ID"
    ].nunique()
)

display(candidates.head())

Rows: 146410
Tracts: 605
Characteristics: 242


,DGUID,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
0,2021S05075350031.00,5350031.00,1402,Total - Indigenous identity for the population...,5470
1,2021S05075350031.00,5350031.00,1411,Total - Registered or Treaty Indian status for...,5470
2,2021S05075350031.00,5350031.00,1414,Total - Private households by tenure - 25% sam...,2670
3,2021S05075350031.00,5350031.00,1416,Renter,2160
4,2021S05075350031.00,5350031.00,1434,Total - Private households by number of person...,2670


In [25]:
print(
    round(
        candidate_path.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

0.31 MB


# Find exact variables

# Helper function

In [26]:
def find_characteristics(
    pattern,
    n=30
):
    result = (
        candidates[
            candidates[
                "CHARACTERISTIC_NAME"
            ]
            .astype(str)
            .str.contains(
                pattern,
                case=False,
                na=False,
                regex=True
            )
        ][
            [
                "CHARACTERISTIC_ID",
                "CHARACTERISTIC_NAME"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            "CHARACTERISTIC_ID"
        )
    )

    return result.head(n)

# Population 

In [27]:
display(
    find_characteristics(
        r"population"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
72,1,"Population, 2021"
240,1066,Total - Other language(s) spoken regularly at ...
241,1078,Total - Knowledge of languages for the populat...
97,111,Total - Income statistics in 2020 for the popu...
109,126,Total - Income statistics in 2020 for the popu...
0,1402,Total - Indigenous identity for the population...
121,141,Total - Employment income statistics in 2020 f...
1,1411,Total - Registered or Treaty Indian status for...
128,148,Composition of total income in 2020 of the pop...
29,1500,Total - Indigenous ancestry for the population...


# Density

In [28]:
display(
    find_characteristics(
        r"density"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
75,6,Population density per square kilometre


# Household income

In [45]:
VARIABLE_IDS = {
    "median_household_income": 243
}

In [46]:
patterns = {
    "population": r"^Population, 2021$",
    "population_density": r"population density",
    "average_household_size": r"average.*household.*size",
    "employment_rate": r"^Employment rate",
    "renter": r"renter",
    "postsecondary": r"postsecondary",
    "transport_warehouse": r"transportation and warehousing",
    "manufacturing": r"manufacturing"
}

for name, pattern in patterns.items():
    print("\n", name.upper())
    display(
        find_characteristics(
            pattern,
            n=30
        )
    )


 POPULATION


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
72,1,"Population, 2021"



 POPULATION_DENSITY


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
75,6,Population density per square kilometre



 AVERAGE_HOUSEHOLD_SIZE


...

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
47,2001,"Postsecondary certificate, diploma or degree"
48,2002,Postsecondary certificate or diploma below...
50,2017,"Postsecondary certificate, diploma or degree"
51,2018,Postsecondary certificate or diploma below...
53,2031,"No postsecondary certificate, diploma or degree"
55,2094,"No postsecondary certificate, diploma or degree"
57,2157,"No postsecondary certificate, diploma or degree"
58,2158,"Postsecondary certificate, diploma or degree"



 TRANSPORT_WAREHOUSE


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
65,2269,48-49 Transportation and warehousing



 MANUFACTURING


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
63,2258,9 Occupations in manufacturing and utilities
64,2266,31-33 Manufacturing


In [51]:
candidates["CHARACTERISTIC_ID"] = pd.to_numeric(
    candidates["CHARACTERISTIC_ID"],
    errors="coerce"
).astype("Int64")

candidates["C1_COUNT_TOTAL"] = pd.to_numeric(
    candidates["C1_COUNT_TOTAL"],
    errors="coerce"
)

In [52]:
income_check = candidates[
    candidates["CHARACTERISTIC_ID"] == 243
][
    [
        "DGUID",
        "CHARACTERISTIC_NAME",
        "C1_COUNT_TOTAL"
    ]
]

print("Rows:", len(income_check))
print("Tracts:", income_check["DGUID"].nunique())

display(income_check.head())

Rows: 605
Tracts: 605


,DGUID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
174,2021S05075350032.00,Median total income of household in 2020 ($),41600.0
416,2021S05075350033.00,Median total income of household in 2020 ($),36400.0
658,2021S05075350034.01,Median total income of household in 2020 ($),62400.0
900,2021S05075350034.02,Median total income of household in 2020 ($),61600.0
1142,2021S05075350035.00,Median total income of household in 2020 ($),66000.0


In [53]:
VARIABLE_IDS = {
    "median_household_income": 243
}

In [54]:
print(
    candidates[
        candidates["CHARACTERISTIC_NAME"]
        .str.contains(
            "Median total income of household",
            case=False,
            na=False
        )
    ][
        [
            "CHARACTERISTIC_ID",
            "CHARACTERISTIC_NAME"
        ]
    ]
    .drop_duplicates()
)

     CHARACTERISTIC_ID                             CHARACTERISTIC_NAME
174                243    Median total income of household in 2020 ($)


In [55]:
income_check = candidates[
    candidates["CHARACTERISTIC_ID"] == 243
][
    [
        "DGUID",
        "CHARACTERISTIC_NAME",
        "C1_COUNT_TOTAL"
    ]
]

print("Rows:", len(income_check))
print("Tracts:", income_check["DGUID"].nunique())

display(income_check.head())

Rows: 605
Tracts: 605


,DGUID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
174,2021S05075350032.00,Median total income of household in 2020 ($),41600.0
416,2021S05075350033.00,Median total income of household in 2020 ($),36400.0
658,2021S05075350034.01,Median total income of household in 2020 ($),62400.0
900,2021S05075350034.02,Median total income of household in 2020 ($),61600.0
1142,2021S05075350035.00,Median total income of household in 2020 ($),66000.0


In [57]:
import pandas as pd

candidate_path = (
    "data/external/statscan/filtered/"
    "gta_census_demographic_candidates.parquet"
)

candidates = pd.read_parquet(candidate_path)

print(candidates.shape)
print(candidates.columns.tolist())

(146410, 5)
['DGUID', 'GEO_NAME', 'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']


In [58]:
candidates["CHARACTERISTIC_ID"] = (
    candidates["CHARACTERISTIC_ID"]
    .astype(str)
    .str.strip()
)

print(
    candidates["CHARACTERISTIC_ID"]
    .isin(["243"])
    .sum()
)

605


In [59]:
income_check = candidates[
    candidates["CHARACTERISTIC_ID"] == "243"
][
    [
        "DGUID",
        "CHARACTERISTIC_ID",
        "CHARACTERISTIC_NAME",
        "C1_COUNT_TOTAL"
    ]
]

print("Rows:", len(income_check))
print("Tracts:", income_check["DGUID"].nunique())

display(income_check.head())

Rows: 605
Tracts: 605


,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
174,2021S05075350032.00,243,Median total income of household in 2020 ($),41600
416,2021S05075350033.00,243,Median total income of household in 2020 ($),36400
658,2021S05075350034.01,243,Median total income of household in 2020 ($),62400
900,2021S05075350034.02,243,Median total income of household in 2020 ($),61600
1142,2021S05075350035.00,243,Median total income of household in 2020 ($),66000


In [60]:
ids = (
    candidates[
        ["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
    ]
    .drop_duplicates()
)

display(
    ids[
        ids["CHARACTERISTIC_ID"]
        .astype(str)
        .str.contains("243", na=False)
    ]
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
174,243,Median total income of household in 2020 ($)


In [61]:
display(
    ids[
        ids["CHARACTERISTIC_NAME"]
        .astype(str)
        .str.contains(
            "income of household",
            case=False,
            na=False
        )
    ].head(50)
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
174,243,Median total income of household in 2020 ($)
175,244,Median after-tax income of household in 2020...
183,252,Average total income of household in 2020 ($)
184,253,Average after-tax income of household in 202...


# Household size

In [30]:
display(
    find_characteristics(
        r"average.*household.*size"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
82,57,Average household size


# Employment

In [31]:
display(
    find_characteristics(
        r"employment rate"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
60,2229,Employment rate
61,2230,Unemployment rate


# Renters

In [32]:
display(
    find_characteristics(
        r"renter"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
3,1416,Renter


# Postsecondary education

In [33]:
display(
    find_characteristics(
        r"postsecondary"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
47,2001,"Postsecondary certificate, diploma or degree"
48,2002,Postsecondary certificate or diploma below...
50,2017,"Postsecondary certificate, diploma or degree"
51,2018,Postsecondary certificate or diploma below...
53,2031,"No postsecondary certificate, diploma or degree"
55,2094,"No postsecondary certificate, diploma or degree"
57,2157,"No postsecondary certificate, diploma or degree"
58,2158,"Postsecondary certificate, diploma or degree"


# Transportation / warehousing workers

In [34]:
display(
    find_characteristics(
        r"transportation and warehousing"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
65,2269,48-49 Transportation and warehousing


# Manufacturing

In [35]:
display(
    find_characteristics(
        r"manufacturing"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
63,2258,9 Occupations in manufacturing and utilities
64,2266,31-33 Manufacturing


# Professional occupations

In [36]:
display(
    find_characteristics(
        r"professional"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
66,2273,"54 Professional, scientific and technical ..."


# Check whether current file has what we need


In [37]:
checks = {
    "population": r"population",
    "density": r"density",
    "income": r"median.*household.*income",
    "household_size": r"average.*household.*size",
    "employment": r"employment rate",
    "renter": r"renter",
    "education": r"postsecondary",
    "transport_warehouse":
        r"transportation and warehousing",
    "manufacturing":
        r"manufacturing"
}

for feature, pattern in checks.items():

    n = (
        candidates[
            "CHARACTERISTIC_NAME"
        ]
        .astype(str)
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
        .sum()
    )

    print(
        f"{feature:22s}",
        "FOUND" if n > 0 else "MISSING"
    )

population             FOUND
density                FOUND
income                 MISSING
household_size         FOUND
employment             FOUND
renter                 FOUND
education              FOUND
transport_warehouse    FOUND
manufacturing          FOUND


# Make an even smaller lookup table

In [38]:
characteristic_lookup = (
    candidates[
        [
            "CHARACTERISTIC_ID",
            "CHARACTERISTIC_NAME"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "CHARACTERISTIC_ID"
    )
)

In [39]:
lookup_path = (
    OUTPUT_DIR /
    "gta_characteristic_lookup.csv"
)

characteristic_lookup.to_csv(
    lookup_path,
    index=False
)

print("Saved:", lookup_path)

Saved: data/external/statscan/filtered/gta_characteristic_lookup.csv


# Validate output

In [40]:
summary = pd.DataFrame({
    "metric": [
        "Walmart catchment tracts",
        "Filtered profile tracts",
        "Candidate characteristics",
        "Filtered rows"
    ],

    "value": [
        len(relevant_tracts),

        candidates[
            "DGUID"
        ].nunique(),

        candidates[
            "CHARACTERISTIC_ID"
        ].nunique(),

        len(candidates)
    ]
})

display(summary)

,metric,value
0,Walmart catchment tracts,605
1,Filtered profile tracts,605
2,Candidate characteristics,242
3,Filtered rows,146410


# Optional: delete candidates you don't need later

In [41]:
pd.read_parquet(
    "data/external/statscan/filtered/"
    "gta_census_demographic_candidates.parquet"
)

,DGUID,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
0,2021S05075350031.00,5350031.00,1402,Total - Indigenous identity for the population...,5470
1,2021S05075350031.00,5350031.00,1411,Total - Registered or Treaty Indian status for...,5470
2,2021S05075350031.00,5350031.00,1414,Total - Private households by tenure - 25% sam...,2670
3,2021S05075350031.00,5350031.00,1416,Renter,2160
4,2021S05075350031.00,5350031.00,1434,Total - Private households by number of person...,2670
...,...,...,...,...,...
146405,2021S05075350528.34,5350528.34,6,Population density per square kilometre,1738.6
146406,2021S05075350528.34,5350528.34,8,Total - Age groups of the population - 100% data,5810
146407,2021S05075350528.34,5350528.34,34,Total - Distribution (%) of the population by ...,100
146408,2021S05075350528.34,5350528.34,39,Average age of the population,40.2


In [42]:
display(
    find_characteristics(
        r"median.*income|income.*household|household.*income",
        n=100
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
97,111,Total - Income statistics in 2020 for the popu...
98,112,Number of total income recipients aged 15 ye...
99,113,Median total income in 2020 among recipien...
100,114,Number of after-tax income recipients aged 1...
101,115,Median after-tax income in 2020 among reci...
...,...,...
229,365,Total - Adjusted after-tax economic family inc...
231,379,Gini index on adjusted household total income
232,380,Gini index on adjusted household market income
233,381,Gini index on adjusted household after-tax i...


In [43]:
income_vars = (
    candidates[
        candidates["CHARACTERISTIC_NAME"]
        .str.contains(
            "income",
            case=False,
            na=False
        )
    ][
        [
            "CHARACTERISTIC_ID",
            "CHARACTERISTIC_NAME"
        ]
    ]
    .drop_duplicates()
    .sort_values("CHARACTERISTIC_ID")
)

display(income_vars.head(100))

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
97,111,Total - Income statistics in 2020 for the popu...
98,112,Number of total income recipients aged 15 ye...
99,113,Median total income in 2020 among recipien...
100,114,Number of after-tax income recipients aged 1...
101,115,Median after-tax income in 2020 among reci...
...,...,...
191,260,Total - Household total income groups in 2020 ...
192,281,Total - Household after-tax income groups in 2...
193,301,Total - Income statistics for economic familie...
194,302,Median total income of economic family in 20...


In [44]:
display(
    income_vars[
        income_vars["CHARACTERISTIC_NAME"]
        .str.contains(
            "household",
            case=False,
            na=False
        )
    ].head(50)
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
97,111,Total - Income statistics in 2020 for the popu...
98,112,Number of total income recipients aged 15 ye...
100,114,Number of after-tax income recipients aged 1...
102,116,Number of market income recipients aged 15 y...
104,118,Number of employment income recipients aged ...
109,126,Total - Income statistics in 2020 for the popu...
110,127,Number of total income recipients aged 15 ye...
112,129,Number of after-tax income recipients aged 1...
114,131,Number of market income recipients aged 15 y...
116,133,Number of employment income recipients aged ...


In [63]:
from pathlib import Path

path = Path(
    "data/external/statscan/filtered/"
    "gta_census_demographic_candidates.parquet"
)

size_mb = path.stat().st_size / 1024**2
size_gb = path.stat().st_size / 1024**3

print("Size MB:", round(size_mb, 2))
print("Size GB:", round(size_gb, 4))

Size MB: 0.31
Size GB: 0.0003
